# sdimg quick visual check

Short, step-by-step sanity test for `sdimg`.
Run top-to-bottom with:
- `../asset/sample_image.png`
- `../asset/sample_mask.png`

In [ ]:
import sys
import tempfile
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from sdimg.fusion import grabcut, otsu_threshold
from sdimg.image import (
    adjust_brightness_contrast,
    clahe_norm,
    decode,
    denoise,
    encode,
    gaussian_blur,
    get_id,
    hist_norm,
    imread,
    imwrite,
    is_image,
    median_blur,
    minmax_norm,
    sharpen,
    to_gray,
    to_rgb,
    zscore_norm,
)
from sdimg.mask import (
    concave_hull,
    convex_hull,
    distance_transform,
    extract_edge,
    fill_holes,
    get_box_from_mask,
    get_box_size,
    get_centroid,
    get_coords,
    get_roi_size,
    is_mask,
    morphology,
    pick_largest,
    to_mask,
    to_roi_box,
)
from sdimg.spatial import crop, flip, merge, pad_to_square, resize, rotate, split

In [ ]:
def display_image(img: np.ndarray) -> np.ndarray:
    if img.ndim == 3 and img.shape[2] == 1:
        return img[..., 0]
    return img


def show_grid(
    items: list[tuple[str, np.ndarray]],
    *,
    cols: int = 3,
    figsize: tuple[int, int] = (15, 10),
    as_mask: bool = False,
) -> None:
    rows = (len(items) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.atleast_1d(axes).ravel()

    for ax in axes:
        ax.axis("off")

    for ax, (title, img) in zip(axes, items):
        ax.set_title(title)
        if as_mask or img.ndim == 2:
            ax.imshow(
                img,
                cmap="gray",
                vmin=0 if as_mask else None,
                vmax=1 if as_mask else None,
            )
        else:
            ax.imshow(display_image(img))

    plt.tight_layout()
    plt.show()


def show_images(items: list[tuple[str, np.ndarray]], **kwargs) -> None:
    show_grid(items, as_mask=False, **kwargs)


def show_masks(items: list[tuple[str, np.ndarray]], **kwargs) -> None:
    show_grid(items, as_mask=True, **kwargs)

In [ ]:
image_path = Path("../asset/sample_image.png")
mask_path = Path("../asset/sample_mask.png")

if not image_path.exists():
    raise FileNotFoundError(f"{image_path} not found")
if not mask_path.exists():
    raise FileNotFoundError(f"{mask_path} not found")

image = imread(image_path)
mask_raw = imread(mask_path)[..., 0]

gray = to_gray(image)
base_mask = to_mask(mask_raw)

print("image:", image.shape, image.dtype)
print("gray:", gray.shape, gray.dtype)
print("mask:", base_mask.shape, base_mask.dtype, np.unique(base_mask))

## Base data


In [ ]:
show_images(
    [
        ("original", image),
        ("gray", gray),
    ]
)
show_masks([("base mask", base_mask)])

## Image utilities


In [ ]:
encoded = encode(image)
decoded = decode(encoded)
image_id = get_id(image, prefix="img_", length=12)

with tempfile.TemporaryDirectory() as tmpdir:
    roundtrip_path = Path(tmpdir) / "roundtrip.png"
    imwrite(roundtrip_path, image)
    roundtrip = imread(roundtrip_path)

print("id:", image_id)
print("encoded chars:", len(encoded))
print("decode matches:", np.array_equal(decoded, image))
print("imwrite/imread matches:", np.array_equal(roundtrip, image))

## Contracts


In [ ]:
rgb_from_gray = to_rgb(gray)
normalized_mask = to_mask(base_mask * 255)

print("is_image(image):", is_image(image))
print("is_mask(base_mask):", is_mask(base_mask))
print("rgb_from_gray:", rgb_from_gray.shape, rgb_from_gray.dtype)
print("normalized_mask unique:", np.unique(normalized_mask))

show_images(
    [
        ("gray", gray),
        ("to_rgb(gray)", rgb_from_gray),
    ]
)

## Image


In [ ]:
image_results = [
    ("zscore_norm", zscore_norm(image)),
    ("hist_norm", hist_norm(image)),
    ("clahe_norm", clahe_norm(image)),
    ("minmax_norm", minmax_norm(image)),
    (
        "brightness_contrast",
        adjust_brightness_contrast(image, brightness=0.1, contrast=0.2),
    ),
    ("gaussian_blur", gaussian_blur(image, (5, 5), 1.2)),
    ("median_blur", median_blur(image, 5)),
    ("denoise", denoise(image)),
    ("sharpen", sharpen(image, alpha=1.0)),
]

show_images([("original", image), *image_results], cols=3, figsize=(15, 16))

## Mask


In [ ]:
mask_ops = [
    ("open", morphology(base_mask, "open", (5, 5), 1)),
    ("close", morphology(base_mask, "close", (5, 5), 1)),
    ("fill_holes", fill_holes(base_mask)),
    ("largest", pick_largest(base_mask)),
    ("convex_hull", convex_hull(base_mask)),
    ("concave_hull", concave_hull(base_mask)),
    ("edge", extract_edge(base_mask)),
]
mask_distance = distance_transform(base_mask)
bbox = get_box_from_mask(base_mask)
roi_box = to_roi_box(base_mask)

print("coords shape:", get_coords(base_mask).shape)
print("bbox:", bbox)
print("area:", get_roi_size(base_mask))
print("box_area:", get_box_size(bbox) if bbox else None)
print("to_roi_box keys:", list(roi_box.keys()) if roi_box else None)
print("centroid:", get_centroid(base_mask))

show_masks([("base", base_mask), *mask_ops], cols=4, figsize=(16, 12))

plt.figure(figsize=(6, 6))
plt.title("distance_transform")
plt.imshow(mask_distance, cmap="magma")
plt.axis("off")
plt.colorbar()
plt.show()

## Spatial


In [ ]:
bbox = get_box_from_mask(base_mask)
if bbox is None:
    raise ValueError("base_mask is empty")

resized = resize(src=image, height=512)
rotated = rotate(src=image, rotation=90)
flipped = flip(src=image, direction="horizontal")
padded, pad_box = pad_to_square(src=image, return_box=True)
cropped = crop(src=image, bbox=bbox)
patches, split_meta = split(src=image, n=(2, 3), overlap=0.25, return_meta=True)
merged = merge(patches=patches, meta=split_meta)

show_images(
    [
        ("original", image),
        ("resize", resized),
        ("rotate 90", rotated),
        ("flip horizontal", flipped),
        ("pad_to_square", padded),
        ("crop", cropped),
        ("merge(split)", merged),
    ],
    cols=3,
    figsize=(15, 14),
)

show_images(
    [(f"patch {i}", p) for i, p in enumerate(patches[:6])], cols=3, figsize=(12, 8)
)

### Paired operations (image + mask)


In [ ]:
bbox = get_box_from_mask(base_mask)
if bbox is None:
    raise ValueError("base_mask is empty")

img_resized = resize(src=image, height=512)
mask_resized = resize(src=base_mask, height=512, interpolation=cv2.INTER_NEAREST)

img_rotated = rotate(src=image, rotation=90)
mask_rotated = rotate(src=base_mask, rotation=90)

img_flipped = flip(src=image, direction="horizontal")
mask_flipped = flip(src=base_mask, direction="horizontal")

img_cropped = crop(src=image, bbox=bbox)
mask_cropped = crop(src=base_mask, bbox=bbox)

show_images(
    [
        ("resize image", img_resized),
        ("rotate image", img_rotated),
        ("flip image", img_flipped),
        ("crop image", img_cropped),
    ],
    cols=2,
    figsize=(12, 10),
)

show_masks(
    [
        ("resize mask", mask_resized),
        ("rotate mask", mask_rotated),
        ("flip mask", mask_flipped),
        ("crop mask", mask_cropped),
    ],
    cols=2,
    figsize=(12, 10),
)

## Fusion


In [ ]:
otsu_mask = otsu_threshold(image=image)

bbox = get_box_from_mask(base_mask)
if bbox is None:
    raise ValueError("base_mask is empty")

wmin, hmin, wmax, hmax = bbox
roi = base_mask[hmin:hmax, wmin:wmax]
grabcut_mask = grabcut(image=image, roi=roi, box=bbox)

show_masks(
    [
        ("otsu", otsu_mask),
        ("initial_roi", roi),
        ("grabcut", grabcut_mask),
    ],
    cols=3,
    figsize=(15, 6),
)

### Grabcut internals


In [ ]:
from matplotlib.colors import BoundaryNorm, ListedColormap
from sdimg.fusion.grabcut import _blur_mask, _build_img, _build_mask, _edge

bbox = get_box_from_mask(base_mask)
if bbox is None:
    raise ValueError("base_mask is empty")

wmin, hmin, wmax, hmax = bbox
roi = base_mask[hmin:hmax, wmin:wmax]
margin = 20

crop_img = crop(src=image, bbox=bbox)
pad_img = cv2.copyMakeBorder(
    crop_img, margin, margin, margin, margin, cv2.BORDER_REFLECT
)
pad_roi = cv2.copyMakeBorder(
    roi, margin, margin, margin, margin, cv2.BORDER_CONSTANT, value=0
)

gray_pad = to_gray(pad_img)
edge_map = _edge(gray=gray_pad)
blur_map = _blur_mask(mask=pad_roi)
feat = _build_img(image=pad_img, roi=pad_roi)
gc_init = _build_mask(roi=pad_roi)
grabcut_mask = grabcut(image=image, roi=roi, box=bbox, margin=margin)

show_images(
    [
        ("crop image", crop_img),
        ("padded image", pad_img),
        ("gray", gray_pad),
        ("edge map", edge_map),
        ("mask blur", blur_map),
        ("feature[0]=gray", feat[..., 0]),
        ("feature[1]=edge", feat[..., 1]),
        ("feature[2]=blur", feat[..., 2]),
    ],
    cols=4,
    figsize=(18, 12),
)

label_names = {
    cv2.GC_BGD: "BGD(0)",
    cv2.GC_FGD: "FGD(1)",
    cv2.GC_PR_BGD: "PR_BGD(2)",
    cv2.GC_PR_FGD: "PR_FGD(3)",
}
vals, cnts = np.unique(gc_init, return_counts=True)
print("grabcut init label counts:")
for v, c in zip(vals.tolist(), cnts.tolist()):
    print(f"  {label_names.get(v, str(v))}: {c}")

cmap = ListedColormap(["black", "lime", "gold", "red"])
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)

fig, axes = plt.subplots(1, 5, figsize=(24, 5))
axes[0].set_title("init labels")
axes[0].imshow(gc_init, cmap=cmap, norm=norm)
axes[0].axis("off")

axes[1].set_title("BGD=0")
axes[1].imshow((gc_init == cv2.GC_BGD).astype(np.uint8), cmap="gray")
axes[1].axis("off")

axes[2].set_title("PR_BGD=2")
axes[2].imshow((gc_init == cv2.GC_PR_BGD).astype(np.uint8), cmap="gray")
axes[2].axis("off")

axes[3].set_title("PR_FGD=3")
axes[3].imshow((gc_init == cv2.GC_PR_FGD).astype(np.uint8), cmap="gray")
axes[3].axis("off")

axes[4].set_title("FGD=1")
axes[4].imshow((gc_init == cv2.GC_FGD).astype(np.uint8), cmap="gray")
axes[4].axis("off")

plt.tight_layout()
plt.show()

show_masks(
    [
        ("roi", roi),
        ("padded roi", pad_roi),
        ("grabcut result", grabcut_mask),
    ],
    cols=3,
    figsize=(14, 5),
)